# 03 — Multi-Mode CatVTON LoRA Fine-Tuning (Adult Prêt + Kids Ethnic + Flared Wedding Lehengas)
- Jointly trains **CatVTON LoRA (`r=16`)** on:
  - Clean zero-leakage **Adult Prêt** (`paired_train` from FYP-2, read-only)
  - Clean zero-leakage **Kids Ethnic & Wedding Formals** (`paired_train` from V3 using our **Dilated Flared Lehenga Masks**)
- Includes the `torchao` Colab bypass, 8-bit AdamW, FP16, and gradient checkpointing (fits comfortably in 16GB T4 VRAM).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers transformers accelerate peft bitsandbytes pillow tqdm


In [ ]:
import sys
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
import peft.import_utils
from peft import LoraConfig, get_peft_model
import bitsandbytes as bnb

# Bypass torchao conflict in Colab peft
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, 'is_torchao_available'):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

device = 'cuda'
V3_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_V3_Kids_Formal_Extension')
FYP2_TRAIN = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research/01_PK_VTON_DATASET/paired_train')
V3_TRAIN = V3_ROOT / '01_KIDS_AND_FORMAL_DATA' / 'paired_train'
LORA_OUT = V3_ROOT / '03_CATVTON_LORA_V3_MULTI_MODE' / 'catvton_v3_universal_lora_r16'
LORA_OUT.mkdir(parents=True, exist_ok=True)

BASE_ID = 'booksforcharlie/stable-diffusion-inpainting'
vae = AutoencoderKL.from_pretrained(BASE_ID, subfolder='vae', torch_dtype=torch.float16).to(device).eval()
vae.requires_grad_(False)

unet = UNet2DConditionModel.from_pretrained(BASE_ID, subfolder='unet', torch_dtype=torch.float16).to(device)
unet.enable_gradient_checkpointing()

lora_cfg = LoraConfig(r=16, lora_alpha=16, target_modules=['to_q', 'to_k', 'to_v', 'to_out.0'], lora_dropout=0.05)
unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()

noise_scheduler = DDPMScheduler.from_pretrained(BASE_ID, subfolder='scheduler')
optimizer = bnb.optim.AdamW8bit(unet.parameters(), lr=1e-4, weight_decay=1e-2)

img_tf = transforms.Compose([transforms.Resize((512, 384)), transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
msk_tf = transforms.Compose([transforms.Resize((512, 384), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

class MultiDomainPKVTONDataset(Dataset):
    def __init__(self, roots):
        self.items = []
        for root, use_flared in roots:
            if not (root / 'person').exists():
                continue
            for p_path in sorted((root / 'person').glob('*.*')):
                stem = p_path.stem
                g_matches = list((root / 'garment').glob(f'{stem}.*'))
                m_folder = (root / 'mask_flared') if (use_flared and (root / 'mask_flared').exists()) else (root / 'mask')
                m_matches = list(m_folder.glob(f'{stem}.*'))
                if g_matches and m_matches:
                    self.items.append((p_path, g_matches[0], m_matches[0]))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        p_p, g_p, m_p = self.items[idx]
        return {
            'person': img_tf(Image.open(p_p).convert('RGB')),
            'garment': img_tf(Image.open(g_p).convert('RGB')),
            'mask': (msk_tf(Image.open(m_p).convert('L')) > 0.5).float(),
        }

ds = MultiDomainPKVTONDataset([(FYP2_TRAIN, False), (V3_TRAIN, True)])
print(f'Joint Multi-Domain Training Samples (Adult Prêt + Kids + Flared Lehengas): {len(ds)}')

loader = DataLoader(ds, batch_size=1, shuffle=True)
GRAD_ACCUM = 8
MAX_STEPS = 1200
step = 0
unet.train()
empty_cond = torch.zeros((1, 77, 768), device=device, dtype=torch.float16)

while step < MAX_STEPS and len(ds) > 0:
    for batch in loader:
        person = batch['person'].to(device, dtype=torch.float16)
        garment = batch['garment'].to(device, dtype=torch.float16)
        mask = batch['mask'].to(device, dtype=torch.float16)
        with torch.no_grad():
            p_lat = vae.encode(person).latent_dist.sample() * vae.config.scaling_factor
            m_lat = vae.encode(person * (1.0 - mask)).latent_dist.sample() * vae.config.scaling_factor
            g_lat = vae.encode(garment).latent_dist.sample() * vae.config.scaling_factor
            msk_lat = F.interpolate(mask, size=p_lat.shape[-2:], mode='nearest')
        target_lat = torch.cat([p_lat, g_lat], dim=-1)
        cond_lat = torch.cat([m_lat, g_lat], dim=-1)
        cond_mask = torch.cat([msk_lat, torch.zeros_like(msk_lat)], dim=-1)
        noise = torch.randn_like(target_lat)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (1,), device=device).long()
        noisy_lat = noise_scheduler.add_noise(target_lat, noise, t)
        pred_noise = unet(torch.cat([noisy_lat, cond_mask, cond_lat], dim=1), t, encoder_hidden_states=empty_cond).sample
        w_half = p_lat.shape[-1]
        loss = F.mse_loss(pred_noise[..., :w_half], noise[..., :w_half]) / GRAD_ACCUM
        loss.backward()
        if (step + 1) % GRAD_ACCUM == 0:
            optimizer.step()
            optimizer.zero_grad()
        step += 1
        if step % 100 == 0:
            print(f'Step {step}/{MAX_STEPS} | loss={loss.item()*GRAD_ACCUM:.4f}')
        if step % 300 == 0 or step >= MAX_STEPS:
            unet.save_pretrained(LORA_OUT / f'step_{step}')
        if step >= MAX_STEPS:
            break
print('Saved Multi-Mode Universal V3 LoRA to:', LORA_OUT)
